In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pickle
from matplotlib import cm
import time
import torch as tc
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, random_split, DataLoader

In [2]:
#This is gratefully borrowed with permission from the notebooks maintained by P. Mehta.

######### LOAD DATA
######### The data consists of 16*10000 samples taken in T=np.arange(0.25,4.0001,0.25):
data_file_name = 'Ising2DFM_reSample_L40_T=All.pkl'
######### The labels are obtained from the following file:
label_file_name = 'Ising2DFM_reSample_L40_T=All_labels.pkl'


############ DATA
with open(data_file_name, 'rb') as pickle_file:
#r=read b=binary pickle must be read in binary mode and needs to be open
# with... as... will automatically close the file after opening it is safer
    data = pickle.load(pickle_file) # pickle reads the file and returns the Python object (1D array, compressed bits) and store in data

#### Decompress array and reshape for convenience
data = np.unpackbits(data).reshape(-1, 1600)
#data has byte (8bits) unpackbits unpack it into 8 bits return a bunch of 0 and 1s
#-1: figure out how many rows there are, each row has 1600=40*40 bits The length of the lattice is 40
data=data.astype('int')
#now convert the datatype to integer

#### map 0 state to -1 (Ising variable can take values +/-1)
data[np.where(data==0)]=-1 
# np.where(data==0) find all indices where data is 0

###### READ LABELS (convention is 1 for ordered states and 0 for disordered states)
with open(label_file_name, 'rb') as pickle_file:
    labels = pickle.load(pickle_file) # pickle reads the file and returns the Python object (here just a 1D array with the binary labels)
print(data.shape) # the shape of the features
print(np.unique(labels)) # the unique labels

(160000, 1600)
[0 1]


In [3]:
X = tc.tensor(data, dtype=tc.float32)
y = tc.tensor(labels, dtype=tc.long)

# reshape to images: (N, 1, 40, 40)
X = X.view(-1, 1, 40, 40)

In [4]:
import torch.nn as nn
import torch.nn.functional as F

class PatchEmbedding(nn.Module):
    def __init__(self, patch_size=10, in_ch=1, embed_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Linear(patch_size*patch_size, embed_dim)

    def forward(self, x):
        # x: (B, 1, 40, 40)
        B, C, H, W = x.shape
        patches = x.unfold(2, self.patch_size, self.patch_size)\
                   .unfold(3, self.patch_size, self.patch_size)
        # patches: (B, 1, num_patches_H, num_patches_W, patch_H, patch_W)
        patches = patches.contiguous().view(B, -1, self.patch_size*self.patch_size)
        # patches: (B, num_patches, patch_size*patch_size)
        embeddings = self.proj(patches)  # (B, num_patches, embed_dim)
        return embeddings

In [5]:
class IsingTransformer(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4, num_layers=2, num_classes=2):
        super().__init__()
        self.patch_embed = PatchEmbedding(embed_dim=embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)  # (B,16,embed_dim)
        x = self.transformer(x)  # (B,16,embed_dim)
        x = x.mean(dim=1)         # global average pooling over patches
        x = self.fc(x)            # (B,num_classes)
        return x

In [8]:
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if tc.cuda.is_available() else "cpu"

dataset = TensorDataset(X, y)
train_size = int(0.8*len(dataset))
test_size = len(dataset) - train_size
train_ds, test_ds = tc.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=8)

model = IsingTransformer().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = tc.optim.Adam(model.parameters(), lr=1e-3)

In [9]:
epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 799.2543
Epoch 2, Loss: 709.5849
Epoch 3, Loss: 484.9335
Epoch 4, Loss: 408.9988
Epoch 5, Loss: 379.9657


In [10]:
model.eval()
correct = 0
total = 0
with tc.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        preds = out.argmax(dim=1)
        correct += (preds==yb).sum().item()
        total += yb.size(0)

print("Test accuracy:", correct/total)

Test accuracy: 0.991625
